# Notebook 01 — Data Ingestion & Feature Engineering

**Input:**
- `public/data/raw/compiled_MASTER_wage_price_analysis.xlsx` — wages, CPI, household size, and 5 core commodity prices
- `public/data/raw/datas/*.csv` — 44 additional PSA commodity price CSVs

**Output:** `public/data/panel.json` — the single master dataset

This notebook:
1. Loads all source sheets from the master spreadsheet
2. Standardizes region names across datasets
3. Loads additional commodity prices from individual PSA CSV files
4. Builds a region × month spine (Jan 2018 – latest available)
5. Forward-fills minimum wages from wage order effective dates
6. Joins all commodity prices, CPI, wages, and household size
7. Computes derived features (days_to_feed, real_wage, rice_share, etc.)
8. Exports to `panel.json`

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

DATA_DIR = Path("../public/data/raw")
MASTER = DATA_DIR / "compiled_MASTER_wage_price_analysis.xlsx"
OUTPUT = Path("../public/data/panel.json")

# Canonical region names used throughout the dashboard
CANONICAL_REGIONS = [
    "NCR", "CAR", "Region I", "Region II", "Region III",
    "Region IV-A", "Region IV-B", "Region V", "Region VI",
    "Region VII", "Region VIII", "Region IX", "Region X",
    "Region XI", "Region XII", "Region XIII", "BARMM",
]

print(f"Master file exists: {MASTER.exists()}")

Master file exists: True


## 1. Region Name Standardization

Each data source uses a different naming convention. We build a mapping function that normalizes all variants to our canonical names.

In [2]:
REGION_MAP = {
    # Commodity price sheets (4a-4e): "NCR - National Capital Region" etc.
    "Philippines": None,
    "NCR - National Capital Region": "NCR",
    "CAR - Cordillera Administrative Region": "CAR",
    "I - Ilocos Region": "Region I",
    "II - Cagayan Valley": "Region II",
    "III - Central Luzon": "Region III",
    "IV-A - CALABARZON": "Region IV-A",
    "MIMAROPA Region": "Region IV-B",
    "V - Bicol Region": "Region V",
    "VI - Western Visayas": "Region VI",
    "VII - Central Visayas": "Region VII",
    "VIII - Eastern Visayas": "Region VIII",
    "IX - Zamboanga Peninsula": "Region IX",
    "X - Northern Mindanao": "Region X",
    "XI - Davao Region": "Region XI",
    "XII - SOCCSKSARGEN": "Region XII",
    "XIII - Caraga": "Region XIII",
    "BARMM - Bangsamoro Autonomous Region in Muslim Mindanao": "BARMM",

    # CPI All Income: "National Capital Region (NCR)" etc.
    "PHILIPPINES": None,
    "National Capital Region (NCR)": "NCR",
    "Areas Outside National Capital Region (AONCR)": None,
    "Cordillera Administrative Region (CAR)": "CAR",
    "Region I (Ilocos Region)": "Region I",
    "Region II (Cagayan Valley)": "Region II",
    "Region III (Central Luzon)": "Region III",
    "Region IV-A (CALABARZON)": "Region IV-A",
    "MIMAROPA Region": "Region IV-B",
    "Region V (Bicol Region)": "Region V",
    "Region VI (Western Visayas)": "Region VI",
    "Region VII (Central Visayas)": "Region VII",
    "Region VIII (Eastern Visayas)": "Region VIII",
    "Region IX (Zamboanga Peninsula)": "Region IX",
    "Region X (Northern Mindanao)": "Region X",
    "Region XI (Davao Region)": "Region XI",
    "Region XII (SOCCSKSARGEN)": "Region XII",
    "Bangsamoro Autonomous Region in Muslim Mindanao (BARMM)": "BARMM",
    "Region XIII (Caraga)": "Region XIII",

    # CPI Bottom 30%: "NCR", "Region 1", etc.
    "AONCR": None,
    "NCR": "NCR",
    "CAR": "CAR",
    "Region 1": "Region I",
    "Region 2": "Region II",
    "Region 3": "Region III",
    "Region 4A": "Region IV-A",
    "MIMAROPA REGION": "Region IV-B",
    "Region 5": "Region V",
    "Region 6": "Region VI",
    "Region 7": "Region VII",
    "Region 8": "Region VIII",
    "Region 9": "Region IX",
    "Region 10": "Region X",
    "Region 11": "Region XI",
    "Region 12": "Region XII",
    "BARMM": "BARMM",
    "CARAGA": "Region XIII",

    # Household size: "NCR (National Capital Region)" etc.
    "NCR (National Capital Region)": "NCR",
    "CAR (Cordillera Administrative Region)": "CAR",
    "REGION I (ILOCOS REGION)": "Region I",
    "REGION II (CAGAYAN VALLEY)": "Region II",
    "REGION III (CENTRAL LUZON)": "Region III",
    "REGION IV-A (CALABARZON)": "Region IV-A",
    "MIMAROPA REGION": "Region IV-B",
    "REGION V (BICOL REGION)": "Region V",
    "REGION VI (WESTERN VISAYAS)": "Region VI",
    "REGION VII (CENTRAL VISAYAS)": "Region VII",
    "REGION VIII (EASTERN VISAYAS)": "Region VIII",
    "REGION IX (ZAMBOANGA PENINSULA)": "Region IX",
    "REGION X (NORTHERN MINDANAO)": "Region X",
    "REGION XI (DAVAO REGION)": "Region XI",
    "REGION XII (SOCCSKSARGEN)": "Region XII",
    "REGION XIII (CARAGA)": "Region XIII",
    "BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO (BARMM) 1": "BARMM",
    "PHILIPPINES (Total)": None,

    # Wage orders: already canonical + special variants
    "Region I": "Region I",
    "Region II": "Region II",
    "Region III": "Region III",
    "Region IV-A": "Region IV-A",
    "Region IV-B": "Region IV-B",
    "Region V": "Region V",
    "Region VI": "Region VI",
    "Region VII": "Region VII",
    "Region VIII": "Region VIII",
    "Region IX": "Region IX",
    "Region X": "Region X",
    "Region XI": "Region XI",
    "Region XII": "Region XII",
    "Region XIII": "Region XIII",
    "CARAGA (XIII)": "Region XIII",
    "ARMM": "BARMM",
}

def normalize_region(name):
    """Map any region name variant to canonical form, or None to drop."""
    if pd.isna(name):
        return None
    name = str(name).strip()
    if name in REGION_MAP:
        return REGION_MAP[name]
    # Try stripping leading dots (PSA CSV format: "..NCR - ...")
    stripped = name.lstrip(". ")
    if stripped in REGION_MAP:
        return REGION_MAP[stripped]
    return None

# Quick test
assert normalize_region("NCR - National Capital Region") == "NCR"
assert normalize_region("CARAGA") == "Region XIII"
assert normalize_region("MIMAROPA REGION") == "Region IV-B"
assert normalize_region("CARAGA (XIII)") == "Region XIII"
assert normalize_region("ARMM") == "BARMM"
print("Region normalization OK")

Region normalization OK


## 2. Load Commodity Prices

Each commodity is in a wide-format sheet: rows = regions, columns = "YYYY Month" prices. We melt each into long format (region, month, price).

In [3]:
def load_commodity_sheet(sheet_name, price_col_name):
    """Load a commodity price sheet from the master XLSX and melt to long format."""
    df = pd.read_excel(MASTER, sheet_name=sheet_name)
    region_col = df.columns[0]

    # Normalize region names, drop non-region rows
    df["region"] = df[region_col].apply(normalize_region)
    df = df.dropna(subset=["region"])

    # Identify monthly price columns (format: "YYYY Month")
    # Skip "Annual" columns
    month_cols = [c for c in df.columns if isinstance(c, str) and
                  any(m in c for m in ["January","February","March","April","May","June",
                                       "July","August","September","October","November","December"])
                  and "Annual" not in c and "Ave" not in c]

    # Melt to long format
    melted = df.melt(id_vars=["region"], value_vars=month_cols,
                     var_name="month_raw", value_name=price_col_name)

    # Parse month_raw ("2024 January") to "2024-01"
    def parse_month(s):
        parts = s.strip().split()
        if len(parts) != 2:
            return None
        year, month_name = parts
        try:
            dt = pd.to_datetime(f"{year} {month_name}", format="%Y %B")
            return dt.strftime("%Y-%m")
        except:
            return None

    melted["month"] = melted["month_raw"].apply(parse_month)
    melted = melted.dropna(subset=["month"])

    # Clean price values: replace ".." and "." with NaN, convert to float
    melted[price_col_name] = pd.to_numeric(melted[price_col_name].replace({"..": np.nan, ".": np.nan}), errors="coerce")

    return melted[["region", "month", price_col_name]]

# Load all five commodities
rice_reg = load_commodity_sheet("4a. Rice (Regular)", "rice_price")
rice_wm = load_commodity_sheet("4b. Rice (Well-Milled)", "rice_wm_price")
eggs = load_commodity_sheet("4c. Eggs", "egg_price")
fish = load_commodity_sheet("4d. Galunggong", "fish_price")
pork = load_commodity_sheet("4e. Pork Kasim", "pork_price")

print(f"Rice regular: {rice_reg.shape}, months: {rice_reg['month'].nunique()}, regions: {rice_reg['region'].nunique()}")
print(f"Rice WM:      {rice_wm.shape}")
print(f"Eggs:         {eggs.shape}")
print(f"Fish:         {fish.shape}")
print(f"Pork:         {pork.shape}")
rice_reg.head()

Rice regular: (1836, 3), months: 108, regions: 17
Rice WM:      (1836, 3)
Eggs:         (1836, 3)
Fish:         (1836, 3)
Pork:         (1836, 3)


,region,month,rice_price
0,NCR,2018-01,37.26
1,CAR,2018-01,41.51
2,Region I,2018-01,35.41
3,Region II,2018-01,37.45
4,Region III,2018-01,38.43


## 2b. Load Additional Commodity Prices from CSV Files

The `public/data/raw/datas/` folder contains PSA price CSVs for additional commodities beyond the five in the master XLSX. These use semicolon delimiters, have a title row + blank row before the header, and use `".."` / `"."` for missing values. We reuse the same region normalization and month parsing, then merge them all into the panel.

In [ ]:
def load_commodity_csv(csv_path, price_col_name):
    """Load a PSA commodity price CSV and melt to long format.

    Format: semicolon-delimited, first row is a title, second row is blank,
    third row is the header. Region names are prefixed with ".." (stripped
    by normalize_region).
    """
    df = pd.read_csv(csv_path, sep=";", skiprows=2, quotechar='"',
                     na_values=[".", "..", ""], keep_default_na=True)
    region_col = df.columns[0]  # "Geolocation"

    df["region"] = df[region_col].apply(normalize_region)
    df = df.dropna(subset=["region"])

    # Identify monthly price columns (format: "YYYY Month"), skip "Annual"
    month_cols = [c for c in df.columns if isinstance(c, str) and
                  any(m in c for m in ["January","February","March","April","May","June",
                                       "July","August","September","October","November","December"])
                  and "Annual" not in c and "Ave" not in c]

    melted = df.melt(id_vars=["region"], value_vars=month_cols,
                     var_name="month_raw", value_name=price_col_name)

    def parse_month(s):
        parts = s.strip().split()
        if len(parts) != 2:
            return None
        year, month_name = parts
        try:
            dt = pd.to_datetime(f"{year} {month_name}", format="%Y %B")
            return dt.strftime("%Y-%m")
        except:
            return None

    melted["month"] = melted["month_raw"].apply(parse_month)
    melted = melted.dropna(subset=["month"])
    melted[price_col_name] = pd.to_numeric(melted[price_col_name], errors="coerce")

    return melted[["region", "month", price_col_name]]


# Mapping: CSV filename → snake_case column name for panel.json
CSV_DIR = DATA_DIR / "datas"

CSV_COMMODITIES = {
    # Cereals & grains
    "psa_corn_grits_white_prices_by_region.csv.csv": "corn_white_price",
    "psa_corn_grits_yellow_prices_by_region.csv.csv": "corn_yellow_price",
    "psa_monggo_green_prices_by_region.csv.csv": "monggo_price",
    # Root crops
    "psa_sweet_patato_(camote)_prices_by_region.csv.csv": "camote_price",
    "psa_cassava_(kamoteng kahoy)_prices_by_region.csv.csv": "cassava_price",
    "psa_gabi_tubers_prices_by_region.csv.csv": "gabi_price",
    "psa_potato_prices_by_region.csv.csv": "potato_price",
    "psa_turnip_(singkamas)_prices_by_region.csv.csv": "singkamas_price",
    # Leafy vegetables
    "psa_kangkong_prices_by_region.csv.csv": "kangkong_price",
    "psa_pechay_native_prices_by_region.csv.csv": "pechay_native_price",
    "psa_pechay_chinese_prices_by_region.csv.csv": "pechay_chinese_price",
    "psa_alugbati_prices_by_region.csv.csv": "alugbati_price",
    "psa_gabi_leaves_prices_by_region.csv.csv": "gabi_leaves_price",
    "psa_malunggay_prices_by_region.csv.csv": "malunggay_price",
    "psa_onion_leaves_prices_by_region.csv.csv": "onion_leaves_price",
    "psa_cabbage_prices_by_region.csv.csv": "cabbage_price",
    # Fruit vegetables
    "psa_tomato_prices_by_region.csv.csv": "tomato_price",
    "psa_eggplant_(talong)_prices_by_region.csv.csv": "talong_price",
    "psa_bitter_gourd_(ampalaya)_prices_by_region.csv.csv": "ampalaya_price",
    "psa_chayote_(sayote)_prices_by_region.csv.csv": "sayote_price",
    "psa_common_gourd_(upo)_prices_by_region.csv.csv": "upo_price",
    "psa_squash_prices_by_region.csv.csv": "squash_price",
    "psa_okra_prices_by_region.csv.csv": "okra_price",
    "psa_cucumber_(pipino)_prices_by_region.csv.csv": "pipino_price",
    "psa_patola_prices_by_region.csv.csv": "patola_price",
    "psa_string_beans_prices_by_region.csv.csv": "string_beans_price",
    "psa_baguio_beans_prices_by_region.csv.csv": "baguio_beans_price",
    # Other vegetables / aromatics
    "psa_carrot_prices_by_region.csv.csv": "carrot_price",
    "psa_raddish_(labanos)_prices_by_region.csv.csv": "labanos_price",
    "psa_onion_red_creole_prices_by_region.csv.csv": "onion_red_price",
    "psa_onion_white_prices_by_region.csv.csv": "onion_white_price",
    "psa_garlic_imported_prices_by_region.csv.csv": "garlic_price",
    "psa_ginger_loose_prices_by_region.csv.csv": "ginger_price",
    # Fish & seafood
    "psa_tilapia_prices_by_region.csv.csv": "tilapia_price",
    "psa_bangus_prices_by_region.csv.csv": "bangus_price",
    "psa_dilis_prices_by_region.csv.csv": "dilis_price",
    "psa_aloy_tulingan_prices_by_region.csv.csv": "tulingan_price",
    "psa_fish_budburon_prices_by_region.csv.csv": "budburon_price",
    "psa_barilis_prices_by_region.csv.csv": "barilis_price",
    "psa_tahong_prices_by_region.csv.csv": "tahong_price",
    "psa_gulaman_dagat_prices_by_region.csv.csv": "gulaman_dagat_price",
    "psa_lato_prices_by_region.csv.csv": "lato_price",
    # Meat & poultry
    "psa_chicken_native_dressed_whole_prices_by_region.csvv.csv": "chicken_native_price",
    "psa_edible_offal_prices_by_region.csv.csv": "edible_offal_price",
}

# Load all CSV commodities
csv_commodities = {}
for filename, col_name in CSV_COMMODITIES.items():
    csv_path = CSV_DIR / filename
    if csv_path.exists():
        csv_commodities[col_name] = load_commodity_csv(csv_path, col_name)
        n = csv_commodities[col_name][col_name].notna().sum()
        print(f"  {col_name}: {csv_commodities[col_name].shape[0]} rows, {n} non-null")
    else:
        print(f"  WARNING: {filename} not found, skipping")

print(f"\nLoaded {len(csv_commodities)} additional commodities from CSV files")

## 3. Load CPI Data

Two CPI series: All Income and Bottom 30%. Both are wide-format with columns like "2018 Jan". We melt to long format (region, month, cpi).

In [4]:
def load_cpi_sheet(sheet_name, value_col_name):
    """Load a CPI sheet (wide format, 'YYYY Mon' columns) and melt to long format."""
    df = pd.read_excel(MASTER, sheet_name=sheet_name)
    region_col = df.columns[0]

    df["region"] = df[region_col].apply(normalize_region)
    df = df.dropna(subset=["region"])

    # Monthly columns: "2018 Jan", "2018 Feb", ... — skip "Ave" columns
    month_cols = [c for c in df.columns if isinstance(c, str)
                  and len(c.split()) == 2
                  and c.split()[0].isdigit()
                  and "Ave" not in c]

    melted = df.melt(id_vars=["region"], value_vars=month_cols,
                     var_name="month_raw", value_name=value_col_name)

    def parse_month(s):
        parts = s.strip().split()
        if len(parts) != 2:
            return None
        year, mon = parts
        try:
            dt = pd.to_datetime(f"{year} {mon}", format="%Y %b")
            return dt.strftime("%Y-%m")
        except:
            return None

    melted["month"] = melted["month_raw"].apply(parse_month)
    melted = melted.dropna(subset=["month"])
    melted[value_col_name] = pd.to_numeric(melted[value_col_name], errors="coerce")

    return melted[["region", "month", value_col_name]]

cpi_all = load_cpi_sheet("3. CPI All Income", "cpi_all")
cpi_b30 = load_cpi_sheet("3. CPI Bottom 30%", "cpi_b30")

print(f"CPI All Income: {cpi_all.shape}, months: {cpi_all['month'].nunique()}")
print(f"CPI Bottom 30%: {cpi_b30.shape}, months: {cpi_b30['month'].nunique()}")
cpi_all.head()

CPI All Income: (1836, 3), months: 108
CPI Bottom 30%: (1836, 3), months: 108


,region,month,cpi_all
0,NCR,2018-01,96.9
1,CAR,2018-01,97.5
2,Region I,2018-01,96.9
3,Region II,2018-01,96.6
4,Region III,2018-01,98.3


## 4. Load Minimum Wages

Wage orders have effective dates and daily rates (some are ranges like "500.00–537.00"). We parse each order, take the **minimum** of the range (non-agricultural floor), then forward-fill to monthly granularity so every region × month has a wage.

In [5]:
def parse_wage(val):
    """Parse wage string: '500.00–537.00' → 500.0, '380.00' → 380.0."""
    s = str(val).strip()
    for sep in ["–", "-", "—"]:
        if sep in s:
            parts = s.split(sep)
            try:
                return min(float(p.strip().replace(",", "")) for p in parts)
            except ValueError:
                return np.nan
    try:
        return float(s.replace(",", ""))
    except ValueError:
        return np.nan

def parse_effective_date(val):
    """Parse 'Nov. 22, 2018' or 'June 4, 2022' to datetime."""
    s = str(val).strip()
    for fmt in ["%b. %d, %Y", "%B %d, %Y", "%b %d, %Y"]:
        try:
            return pd.to_datetime(s, format=fmt)
        except:
            continue
    return pd.NaT

wages_raw = pd.read_excel(MASTER, sheet_name="1. Min Wages by Region")
wages_raw["region"] = wages_raw["Region"].apply(normalize_region)
wages_raw = wages_raw.dropna(subset=["region"])
wages_raw["eff_date"] = wages_raw["Effective Date"].apply(parse_effective_date)
wages_raw["daily_wage"] = wages_raw["Daily Rate Non-Agri (₱)"].apply(parse_wage)
wages_raw = wages_raw.dropna(subset=["eff_date", "daily_wage"])
wages_raw["eff_month"] = wages_raw["eff_date"].dt.to_period("M")

print(f"Wage orders loaded: {len(wages_raw)} orders across {wages_raw['region'].nunique()} regions")
print(f"Wage range: ₱{wages_raw['daily_wage'].min():.0f} – ₱{wages_raw['daily_wage'].max():.0f}")
wages_raw[["region", "eff_date", "daily_wage"]].head(10)

Wage orders loaded: 128 orders across 17 regions
Wage range: ₱256 – ₱658


,region,eff_date,daily_wage
0,NCR,2018-11-22,500.0
1,NCR,2022-06-04,533.0
2,NCR,2023-07-16,573.0
3,NCR,2024-07-17,608.0
4,NCR,2025-07-18,658.0
5,CAR,2018-08-20,300.0
6,CAR,2019-11-18,340.0
7,CAR,2022-06-14,380.0
8,CAR,2023-01-01,400.0
9,CAR,2023-12-05,430.0


## 5. Load Household Size

In [6]:
hh_raw = pd.read_excel(MASTER, sheet_name="6. Household Size 2020")
hh_raw["region"] = hh_raw["Region"].apply(normalize_region)
hh_raw = hh_raw.dropna(subset=["region"])
hh_size = hh_raw[["region", "Average Household Size"]].rename(
    columns={"Average Household Size": "household_size"}
)
hh_size["household_size"] = pd.to_numeric(hh_size["household_size"], errors="coerce")

print(f"Household sizes for {len(hh_size)} regions")
print(f"Range: {hh_size['household_size'].min():.2f} – {hh_size['household_size'].max():.2f}")
hh_size

Household sizes for 17 regions
Range: 3.83 – 5.93


,region,household_size
0,NCR,3.83
1,CAR,4.08
2,Region I,4.05
3,Region II,4.05
4,Region III,4.07
5,Region IV-A,3.97
6,Region IV-B,4.05
7,Region V,4.44
8,Region VI,4.09
9,Region VII,4.09


## 6. Build Region × Month Spine & Forward-Fill Wages

Create the master spine (17 regions × all months), then forward-fill wage orders so every cell has the applicable minimum wage.

In [7]:
# Determine date range from commodity data
all_months_available = sorted(set(rice_reg["month"]) | set(cpi_all["month"]))
start_month = min(all_months_available)
end_month = max(all_months_available)
print(f"Date range: {start_month} to {end_month}")

# Build spine: every region × every month
months = pd.date_range(start_month, end_month, freq="MS").strftime("%Y-%m").tolist()
spine = pd.DataFrame(
    [(r, m) for r in CANONICAL_REGIONS for m in months],
    columns=["region", "month"]
)
print(f"Spine shape: {spine.shape} ({len(CANONICAL_REGIONS)} regions × {len(months)} months)")

# Forward-fill wages onto the spine
# For each region, place wage orders at their effective month, then forward-fill
def build_wage_series(region_orders, months):
    """Given a region's wage orders, produce a monthly series via forward-fill."""
    series = pd.Series(index=months, dtype=float)
    for _, row in region_orders.iterrows():
        m = row["eff_month"].strftime("%Y-%m")
        if m in series.index:
            series[m] = row["daily_wage"]
    series = series.ffill()
    return series

wage_monthly = []
for region in CANONICAL_REGIONS:
    region_orders = wages_raw[wages_raw["region"] == region].sort_values("eff_date")
    ws = build_wage_series(region_orders, months)
    for m, w in ws.items():
        wage_monthly.append({"region": region, "month": m, "daily_wage": w})

wage_df = pd.DataFrame(wage_monthly)
print(f"Wage monthly: {wage_df.shape}, missing: {wage_df['daily_wage'].isna().sum()}")

# Backfill: for months before the first wage order, use the earliest known wage
wage_df["daily_wage"] = wage_df.groupby("region")["daily_wage"].bfill()

print(f"After backfill, missing: {wage_df['daily_wage'].isna().sum()}")
wage_df.head()

Date range: 2018-01 to 2026-12
Spine shape: (1836, 2) (17 regions × 108 months)
Wage monthly: (1836, 3), missing: 129
After backfill, missing: 0


,region,month,daily_wage
0,NCR,2018-01,500.0
1,NCR,2018-02,500.0
2,NCR,2018-03,500.0
3,NCR,2018-04,500.0
4,NCR,2018-05,500.0


## 7. Merge All Datasets

Join commodity prices, CPI, wages, and household size onto the spine.

In [ ]:
panel = spine.copy()

# Merge commodity prices (from master XLSX)
for df, name in [(rice_reg, "rice_price"), (rice_wm, "rice_wm_price"),
                 (eggs, "egg_price"), (fish, "fish_price"), (pork, "pork_price")]:
    panel = panel.merge(df, on=["region", "month"], how="left")
    print(f"After {name}: {panel[name].notna().sum()} / {len(panel)} non-null")

# Merge additional commodity prices (from CSV files)
for col_name, df in csv_commodities.items():
    panel = panel.merge(df, on=["region", "month"], how="left")
    n = panel[col_name].notna().sum()
    print(f"After {col_name}: {n} / {len(panel)} non-null")

# Merge CPI
panel = panel.merge(cpi_all, on=["region", "month"], how="left")
panel = panel.merge(cpi_b30, on=["region", "month"], how="left")
print(f"CPI all: {panel['cpi_all'].notna().sum()} / {len(panel)} non-null")
print(f"CPI b30: {panel['cpi_b30'].notna().sum()} / {len(panel)} non-null")

# Merge wages
panel = panel.merge(wage_df, on=["region", "month"], how="left")
print(f"Wages: {panel['daily_wage'].notna().sum()} / {len(panel)} non-null")

# Merge household size (static — broadcast across all months)
panel = panel.merge(hh_size, on="region", how="left")
print(f"Household size: {panel['household_size'].notna().sum()} / {len(panel)} non-null")

print(f"\nFinal panel shape: {panel.shape}")
print(f"Columns: {list(panel.columns)}")
panel.head()

## 8. Compute Derived Features

Key metrics:
- **`days_to_feed`**: (daily basket cost per person × household size × 30) / (daily wage × 30) = daily basket cost × household size / daily wage
- **`real_wage`**: daily wage deflated by CPI (base 2018 = 100)
- **`rice_share`**: fraction of basket cost from rice
- **`wage_basket_gap`**: daily wage minus daily household basket cost

In [ ]:
# Default food basket (per person per day, in kg or units)
# Matches DEFAULT_BASKET in src/lib/constants.ts
# Based on PSA Food Threshold Menu (government's official food poverty line)
BASKET = {
    "rice_price": 0.35,          # 350g regular rice
    "monggo_price": 0.03,        # 30g monggo
    "camote_price": 0.05,        # 50g camote
    "kangkong_price": 0.05,      # 50g kangkong
    "malunggay_price": 0.03,     # 30g malunggay
    "fish_price": 0.08,          # 80g galunggong
    "dilis_price": 0.02,         # 20g dilis
    "egg_price": 1,              # 1 pc egg
    "pork_price": 0.05,          # 50g pork kasim
}

# Daily basket cost per person (sum of price × quantity for each item)
panel["daily_basket_pp"] = sum(
    panel[col] * qty for col, qty in BASKET.items()
)

# Daily basket cost for the household
panel["daily_basket_hh"] = panel["daily_basket_pp"] * panel["household_size"]

# Monthly basket cost
panel["monthly_basket"] = panel["daily_basket_hh"] * 30

# Days of minimum wage to feed household for 30 days
# = monthly_basket / daily_wage = (daily_basket_hh * 30) / daily_wage
panel["days_to_feed"] = panel["monthly_basket"] / panel["daily_wage"]

# Real wage (deflated by CPI all income, base period = 100)
panel["real_wage"] = panel["daily_wage"] / panel["cpi_all"] * 100

# Rice share of basket cost
rice_cost = panel["rice_price"] * BASKET["rice_price"]
panel["rice_share"] = rice_cost / panel["daily_basket_pp"]

# Wage–basket gap (daily surplus/deficit)
panel["wage_basket_gap"] = panel["daily_wage"] - panel["daily_basket_hh"]

print("Derived features computed.")
print(f"days_to_feed range: {panel['days_to_feed'].min():.1f} – {panel['days_to_feed'].max():.1f}")
print(f"real_wage range: {panel['real_wage'].min():.0f} – {panel['real_wage'].max():.0f}")
print(f"rice_share range: {panel['rice_share'].min():.2f} – {panel['rice_share'].max():.2f}")
panel[["region", "month", "daily_wage", "daily_basket_pp", "daily_basket_hh",
       "days_to_feed", "real_wage", "rice_share"]].head(10)

## 9. Data Quality Check

In [ ]:
# Drop rows where we can't compute the core metric
# Note: egg_price excluded — Region VII (Central Visayas) has no egg data in the source
core_cols = ["daily_wage", "rice_price", "fish_price", "pork_price", "household_size"]
before = len(panel)
panel = panel.dropna(subset=core_cols)
after = len(panel)
print(f"Dropped {before - after} rows missing core columns ({before} → {after})")

# Fill missing egg prices with national median for that month (for basket computation)
egg_medians = panel.groupby("month")["egg_price"].median()
panel["egg_price"] = panel.apply(
    lambda r: egg_medians[r["month"]] if pd.isna(r["egg_price"]) else r["egg_price"],
    axis=1
)

# Fill missing prices for new basket items with national median for that month
for col in ["monggo_price", "camote_price", "kangkong_price", "malunggay_price", "dilis_price"]:
    if col in panel.columns:
        medians = panel.groupby("month")[col].median()
        panel[col] = panel.apply(
            lambda r, c=col, m=medians: m[r["month"]] if pd.isna(r[c]) else r[c],
            axis=1
        )

# Recompute derived features after fill
panel["daily_basket_pp"] = sum(
    panel[col] * qty for col, qty in BASKET.items()
)
panel["daily_basket_hh"] = panel["daily_basket_pp"] * panel["household_size"]
panel["monthly_basket"] = panel["daily_basket_hh"] * 30
panel["days_to_feed"] = panel["monthly_basket"] / panel["daily_wage"]
rice_cost = panel["rice_price"] * BASKET["rice_price"]
panel["rice_share"] = rice_cost / panel["daily_basket_pp"]
panel["wage_basket_gap"] = panel["daily_wage"] - panel["daily_basket_hh"]

# Summary of remaining nulls
null_counts = panel.isnull().sum()
print("\nRemaining nulls per column:")
print(null_counts[null_counts > 0].to_string() if null_counts.any() else "None")

# Coverage summary
print(f"\nFinal: {panel['region'].nunique()} regions, {panel['month'].nunique()} months")
print(f"Date range: {panel['month'].min()} to {panel['month'].max()}")
panel.describe().round(2)

## 10. Export to `panel.json`

Write the final panel as a JSON array of objects, ready for the dashboard.

In [11]:
# Round numeric columns for smaller JSON output
numeric_cols = panel.select_dtypes(include=[np.number]).columns
panel[numeric_cols] = panel[numeric_cols].round(4)

# Replace NaN with None for clean JSON
records = json.loads(panel.to_json(orient="records"))

OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT, "w") as f:
    json.dump(records, f, indent=2)

file_size_mb = OUTPUT.stat().st_size / (1024 * 1024)
print(f"Exported {len(records)} rows to {OUTPUT}")
print(f"File size: {file_size_mb:.2f} MB")
print(f"Columns: {list(panel.columns)}")
print("\nSample record:")
print(json.dumps(records[0], indent=2))

Exported 1700 rows to ..\public\data\panel.json
File size: 0.84 MB
Columns: ['region', 'month', 'rice_price', 'rice_wm_price', 'egg_price', 'fish_price', 'pork_price', 'cpi_all', 'cpi_b30', 'daily_wage', 'household_size', 'daily_basket_pp', 'daily_basket_hh', 'monthly_basket', 'days_to_feed', 'real_wage', 'rice_share', 'wage_basket_gap']

Sample record:
{
  "region": "NCR",
  "month": "2018-01",
  "rice_price": 37.26,
  "rice_wm_price": 43.8,
  "egg_price": 5.77,
  "fish_price": 159.84,
  "pork_price": 262.34,
  "cpi_all": 96.9,
  "cpi_b30": 96.7,
  "daily_wage": 500.0,
  "household_size": 3.83,
  "daily_basket_pp": 86.8,
  "daily_basket_hh": 332.444,
  "monthly_basket": 9973.32,
  "days_to_feed": 19.9466,
  "real_wage": 515.9959,
  "rice_share": 0.1717,
  "wage_basket_gap": 167.556
}
